In [ ]:
%%writefile PrismTrainKaggle.py

import os
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader
from torch.utils.data.distributed import DistributedSampler
from torchsummary import summary
import torch.distributed as dist
import torch.backends.cudnn as TBC
from torch.nn.parallel import DistributedDataParallel as DDP
import cv2
import numpy as np
import time
import math
from pathlib import Path
import wandb
import contextlib

torch._inductor.config.max_autotune = False

torch.manual_seed(7)
torch.cuda.manual_seed_all(7)
TBC.benchmark = True
TBC.deterministic = False
cv2.setNumThreads(0)
StartTime = time.time()
MaxTime = 11.5*3600

NumHeads = 8
HeadDims = 32
EmbedDim = 256

MPDType = torch.float16

BatchSize = 16
TargetLR = 1e-4
WarmupSteps = 1000

TrainDirName = '/kaggle/temp/prismpreprocessed/train'
ValDirName = '/kaggle/temp/prismpreprocessed/val'

WANDB_PROJECT = user_secrets.get_secret('WANDB_PROJECT')
WANDB_ENTITY = user_secrets.get_secret('WANDB_ENTITY')
WANDB_ARTIFACT_NAME = 'prism-checkpoint'
WANDB_ARTIFACT_DIR = Path('/kaggle/working/wandb_resume')
LATEST_CHECKPOINT_PATH = Path('/kaggle/working/PrismModelLatest.pth')
BEST_CHECKPOINT_PATH = Path('/kaggle/working/PrismModelBest.pth')

class ImageDataset(Dataset):
	def __init__(self, BaseDir, IsTraining):
		super().__init__()
		self.IsTraining = IsTraining
		
		LDir = os.path.join(BaseDir, 'L')
		ABDir = os.path.join(BaseDir, 'ab')
		
		ABStems = {os.path.splitext(f)[0] for f in os.listdir(ABDir) if f.endswith('.npy')}
		CommonStems = sorted(os.path.splitext(f)[0] for f in os.listdir(LDir) if f.endswith('.jpg') and os.path.splitext(f)[0] in ABStems)

		self.ImgPaths = [(os.path.join(LDir, s + '.jpg'), os.path.join(ABDir, s + '.npy')) for s in CommonStems]

	def __len__(self):
		return len(self.ImgPaths)
	
	def __getitem__(self, index):
		try:
			InputImg = cv2.imread(self.ImgPaths[index][0], cv2.IMREAD_GRAYSCALE)
			if InputImg is None:
				raise ValueError(f"OpenCV could not read image: {self.ImgPaths[index][0]}")
			
			OutputIndices = np.load(self.ImgPaths[index][1])

			if self.IsTraining:
				if torch.rand(1).item() < 0.5:
					InputImg = cv2.flip(InputImg, 1)
					OutputIndices = np.flip(OutputIndices, 1)

			InputChannel = (torch.from_numpy(InputImg).float() / 255.0).unsqueeze(2).permute(2, 0, 1)
			OutputTensor = torch.from_numpy(OutputIndices.copy()).long()

			return (InputChannel, OutputTensor)
		except (ValueError,OSError,EOFError):
			RandomFallback = torch.randint(0, len(self.ImgPaths), (1,)).item()
			return self.__getitem__(RandomFallback)

class Attention(nn.Module):
	def __init__(self):
		super().__init__()

		self.QueryMatrix = nn.Linear(EmbedDim,EmbedDim)
		self.KeyMatrix = nn.Linear(EmbedDim,EmbedDim)
		self.ValueMatrix = nn.Linear(EmbedDim,EmbedDim)
		self.OutputMatrix = nn.Linear(EmbedDim,EmbedDim)

	def forward(self,x):
		# SeqLen = 256 currently
		BatchSize,SeqLen,_ = x.shape

		QueryOutput = self.QueryMatrix(x)
		KeyOutput = self.KeyMatrix(x)
		ValueOutput = self.ValueMatrix(x)

		# 8 Heads
		QueryOutput = QueryOutput.view(BatchSize,SeqLen,NumHeads,HeadDims).transpose(1,2) # B,NumHeads,256,HeadDims
		KeyOutput = KeyOutput.view(BatchSize,SeqLen,NumHeads,HeadDims).transpose(1,2) # B,NumHeads,256,HeadDims
		ValueOutput = ValueOutput.view(BatchSize,SeqLen,NumHeads,HeadDims).transpose(1,2) # B,NumHeads,256,HeadDims

		#Attention(Q, K, V) = (softmax((QK^T)/sqrt(dk)))*V, dk = HeadDims
		AttentionOutput = torch.nn.functional.scaled_dot_product_attention(QueryOutput, KeyOutput, ValueOutput)
		AttentionOutput = AttentionOutput.transpose(1,2).contiguous().view(BatchSize,SeqLen,EmbedDim)

		FinalOutput = self.OutputMatrix(AttentionOutput)

		return FinalOutput

class PrismBlock(nn.Module):
	def __init__(self):
		super().__init__()

		self.PseudoQuery = nn.Parameter(torch.randn(EmbedDim)*0.02)
		self.Attention = Attention()
		self.MultiLayerPerceptron = nn.Sequential(
			nn.Linear(in_features=EmbedDim,out_features=EmbedDim*4),
			nn.GELU(),
			nn.Linear(in_features=EmbedDim*4,out_features=EmbedDim)
		)
		self.LayerNorm1 = nn.LayerNorm(EmbedDim)
		self.LayerNorm2 = nn.LayerNorm(EmbedDim)
		self.LayerNorm3 = nn.LayerNorm(EmbedDim)
		self.Softmax = nn.Softmax(dim=0)

	def forward(self,x,HistoryList):
		HistoryList = torch.stack(HistoryList,dim=0) # D,B,SeqLen,EmbedDim
		HistoryListNorm = self.LayerNorm1(HistoryList)

		Scores = (HistoryListNorm*self.PseudoQuery).sum(dim=-1)/math.sqrt(EmbedDim) # D,B,SeqLen
		Weights = self.Softmax(Scores).unsqueeze(-1)

		XRouted = (HistoryList*Weights).sum(dim=0)

		XRoutedNorm = self.LayerNorm2(XRouted)
		x = x + self.Attention(XRoutedNorm)

		xNorm = self.LayerNorm3(x)
		x = x + self.MultiLayerPerceptron(xNorm)

		return x

class PrismModel(nn.Module):
	def __init__(self):
		super().__init__()
		self.Stem = nn.Sequential(
			nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1),
			nn.GroupNorm(8, 64),
			nn.GELU()
		)
		self.Stage1 = nn.Sequential(
			nn.Conv2d(64, 96, kernel_size=3, stride=2, padding=1), # 256 -> 128
			nn.GroupNorm(16, 96),
			nn.GELU(),
			nn.Conv2d(96, 96, kernel_size=3, stride=1, padding=1),
			nn.GroupNorm(16, 96),
			nn.GELU()
		)
		self.Stage2 = nn.Sequential(
			nn.Conv2d(96, 128, kernel_size=3, stride=2, padding=1), # 128 -> 64
			nn.GroupNorm(16, 128),
			nn.GELU(),
			nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1),
			nn.GroupNorm(16, 128),
			nn.GELU()
		)
		self.Stage3 = nn.Sequential(
			nn.Conv2d(128, 192, kernel_size=3, stride=2, padding=1), # 64 -> 32
			nn.GroupNorm(16, 192),
			nn.GELU(),
			nn.Conv2d(192, 192, kernel_size=3, stride=1, padding=1),
			nn.GroupNorm(16, 192),
			nn.GELU()
		)
		self.PatchEmbed = nn.Sequential(
			nn.Conv2d(192, 256, kernel_size=2, stride=2), # 32 -> 16
		)
		
		self.Flatten = nn.Flatten(start_dim=2) # B, 256, 256
		self.LearnParam = nn.Parameter(torch.randn((1, 256, 256)) * 0.02)
		
		self.ModuleList = nn.ModuleList([PrismBlock() for _ in range(8)])
		self.FinalNorm = nn.LayerNorm(EmbedDim)
		
		self.Decoder3 = nn.Sequential(
			nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False), # 16 -> 32
			nn.Conv2d(256 + 192, 192, kernel_size=3, stride=1, padding=1),
			nn.GroupNorm(16, 192),
			nn.GELU()
		)
		self.Decoder2 = nn.Sequential(
			nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False), # 32 -> 64
			nn.Conv2d(192 + 128, 128, kernel_size=3, stride=1, padding=1),
			nn.GroupNorm(16, 128),
			nn.GELU()
		)
		self.Decoder1 = nn.Sequential(
			nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False), # 64 -> 128
			nn.Conv2d(128 + 96, 96, kernel_size=3, stride=1, padding=1),
			nn.GroupNorm(16, 96),
			nn.GELU()
		)
		self.Decoder0 = nn.Sequential(
			nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False), # 128 -> 256
			nn.Conv2d(96 + 64, 64, kernel_size=3, stride=1, padding=1),
			nn.GroupNorm(8, 64),
			nn.GELU(),
			nn.Conv2d(64, 313, kernel_size=1, stride=1)
		)

	def forward(self, x):
		Skip0 = self.Stem(x)          # B, 64, 256, 256
		Skip1 = self.Stage1(Skip0)    # B, 96, 128, 128
		Skip2 = self.Stage2(Skip1)    # B, 128, 64, 64
		Skip3 = self.Stage3(Skip2)    # B, 192, 32, 32
		
		Tokens = self.PatchEmbed(Skip3) # B, 256, 16, 16
		Tokens = self.Flatten(Tokens)   # B, 256, 256
		Tokens = Tokens.transpose(1, 2) + self.LearnParam # B, 256, 256

		HistoryList = [Tokens]
		for module in self.ModuleList:
			Tokens = module(Tokens, HistoryList)
			HistoryList.append(Tokens)

		Tokens = self.FinalNorm(Tokens)
		x = Tokens.transpose(1, 2).reshape(-1, 256, 16, 16)

		x = torch.cat([self.Decoder3[0](x), Skip3], dim=1)
		x = self.Decoder3[1:](x)
		
		x = torch.cat([self.Decoder2[0](x), Skip2], dim=1)
		x = self.Decoder2[1:](x)
		
		x = torch.cat([self.Decoder1[0](x), Skip1], dim=1)
		x = self.Decoder1[1:](x)
		
		x = torch.cat([self.Decoder0[0](x), Skip0], dim=1)
		x = self.Decoder0[1:](x)

		return x

PriorProbs = np.load('/kaggle/input/datasets/nkminion/prismtrain/PriorProbs.npy')
Lambda = 0.5
Q = 313.0
SmoothedProbs = ((1.0-Lambda)*PriorProbs) + (Lambda/Q)
Weights = 1.0/SmoothedProbs
Weights = np.power(Weights, 0.5)
NormalisationFactor = np.sum(PriorProbs*Weights)
Weights /= NormalisationFactor
ClassWeightsTensor = torch.from_numpy(np.ascontiguousarray(Weights)).float().reshape(-1)
assert ClassWeightsTensor.numel() == 313, f"class weights must have 313 entries, got {ClassWeightsTensor.numel()}"

Buckets = np.load('/kaggle/input/datasets/nkminion/prismtrain/CoordBuckets.npy')
BucketsT = torch.from_numpy(Buckets).float()
DistMat = torch.cdist(BucketsT, BucketsT, p=2) ** 2
SqDist, SoftIdx = torch.topk(DistMat, k=5, largest=False, dim=-1)
sigma = 5.0
SoftWeights = torch.exp(-SqDist / (2.0 * (sigma ** 2)))
SoftWeights = SoftWeights / SoftWeights.sum(dim=-1, keepdim=True)

def SoftCrossEntropyLoss(predictions, targets, SoftIdx, SoftWeights, ClassWeightsTensor):
	device = predictions.device
	SoftIdx = SoftIdx.to(device)
	SoftWeights = SoftWeights.to(device)
	ClassWeightsTensor = ClassWeightsTensor.to(device)
	
	NeighborIndices = SoftIdx[targets].permute(0, 3, 1, 2).contiguous().long()
	NeighborWeights = SoftWeights[targets].permute(0, 3, 1, 2).contiguous()
	
	GatheredLogits = torch.gather(predictions, 1, NeighborIndices).float()
	LogZ = torch.logsumexp(predictions.float(), dim=1, keepdim=True)
	GatheredLogProbs = GatheredLogits - LogZ

	PixelLoss = -(NeighborWeights * GatheredLogProbs).sum(dim=1)
	PixelWeights = ClassWeightsTensor[targets]
	
	return (PixelLoss * PixelWeights).mean()


def InitWandbRun(rank):
	if rank != 0:
		return None

	InitKwargs = {
		'project': WANDB_PROJECT,
		'resume': 'allow',
		'config': {
			'BatchSize': BatchSize,
			'TargetLR': TargetLR,
			'WarmupSteps': WarmupSteps,
			'NumHeads': NumHeads,
			'HeadDims': HeadDims,
			'EmbedDim': EmbedDim
		}
	}

	if WANDB_ENTITY:
		InitKwargs['entity'] = WANDB_ENTITY

	RunName = os.environ.get('WANDB_RUN_NAME')
	if RunName:
		InitKwargs['name'] = RunName

	try:
		return wandb.init(**InitKwargs)
	except Exception as Error:
		print(f'W&B init failed. Continuing without W&B checkpoint sync. Error: {Error}')
		return None


def DownloadLatestCheckpoint(wandb_run):
	if wandb_run is None:
		return None

	try:
		Artifact = wandb_run.use_artifact(f'{WANDB_ARTIFACT_NAME}:latest', type='model')
		ArtifactDir = Path(Artifact.download(root=str(WANDB_ARTIFACT_DIR)))
		CheckpointPath = ArtifactDir / LATEST_CHECKPOINT_PATH.name

		if CheckpointPath.exists():
			print(f'Downloaded W&B latest checkpoint: {CheckpointPath}')
			return CheckpointPath

		Candidates = list(ArtifactDir.glob('*.pth'))
		if Candidates:
			print(f'Downloaded W&B checkpoint: {Candidates[0]}')
			return Candidates[0]

		print('W&B latest artifact downloaded, but no checkpoint file was found.')
		return None
	except Exception as Error:
		print(f'No W&B latest checkpoint found. Starting from scratch. Error: {Error}')
		return None

def CleanName(n):
	for pre in ('_orig_mod.', '_fsdp_wrapped_module.', '_checkpoint_wrapped_module.', 'module.'):
		n = n.replace(pre, '')
	return n

def CleanCheckpointStateDict(RawStateDict):
	return {CleanName(k): v for k, v in RawStateDict.items()}


def LoadCheckpointModelState(model, RawStateDict):
	LoadErrors = []
	for CandidateStateDict in (RawStateDict, CleanCheckpointStateDict(RawStateDict)):
		try:
			model.load_state_dict(CandidateStateDict)
			return
		except RuntimeError as Error:
			LoadErrors.append(str(Error))

	raise RuntimeError('Could not load checkpoint model state. Last error: ' + LoadErrors[-1])


def BuildCheckpoint(model, optimiser, scheduler, ImagesSeen, BestLoss, TrainLossHist, ValLossHist, StepCounter, Epoch, PatienceCounter, scaler):
	if int(os.environ['LOCAL_RANK']) == 0:
		RawModel = model.module if hasattr(model, 'module') else model
		return {
			'ModelState': RawModel.state_dict(),
			'OptimiserState': optimiser.state_dict(),
			'SchedulerState': scheduler.state_dict(),
			'ScalerState': scaler.state_dict(),
			'ImagesSeen': ImagesSeen,
			'BestLoss': BestLoss,
			'TrainLossHist': TrainLossHist,
			'ValLossHist': ValLossHist,
			'StepCounter': StepCounter,
			'Epoch': Epoch,
			'PatienceCounter': PatienceCounter
		}
	return None


def SaveCheckpoint(CheckpointPath, Checkpoint):
	CheckpointPath.parent.mkdir(parents=True, exist_ok=True)
	torch.save(Checkpoint, CheckpointPath)


def UploadCheckpointArtifact(wandb_run, CheckpointPath, Aliases, Metadata):
	if wandb_run is None:
		return

	try:
		Artifact = wandb.Artifact(WANDB_ARTIFACT_NAME, type='model', metadata=Metadata)
		Artifact.add_file(str(CheckpointPath), name=CheckpointPath.name)
		LoggedArtifact = wandb_run.log_artifact(Artifact, aliases=Aliases)
		LoggedArtifact.wait()
	except Exception as Error:
		print(f'W&B artifact upload failed for {CheckpointPath}. Error: {Error}')


def GetCurrentLR(optimiser):
	return optimiser.param_groups[0]['lr']


def DDPTrainLoop(Weights,BatchSize,TrainDirName,ValDirName):
	rank = int(os.environ['LOCAL_RANK'])
	GPUCount = int(os.environ['WORLD_SIZE'])
	dist.init_process_group('nccl')
	torch.cuda.set_device(rank)
	device = torch.device(f'cuda:{rank}')
	wandb_run = InitWandbRun(rank)

	ResumeCheckpointPath = None
	if rank == 0:
		ResumeCheckpointPath = DownloadLatestCheckpoint(wandb_run)
	dist.barrier()

	if ResumeCheckpointPath is None:
		for CandidatePath in (WANDB_ARTIFACT_DIR / LATEST_CHECKPOINT_PATH.name, LATEST_CHECKPOINT_PATH):
			if CandidatePath.exists():
				ResumeCheckpointPath = CandidatePath
				break
		if ResumeCheckpointPath is None and WANDB_ARTIFACT_DIR.exists():
			Candidates = list(WANDB_ARTIFACT_DIR.glob('*.pth'))
			if Candidates:
				ResumeCheckpointPath = Candidates[0]

	model = PrismModel().to(device, non_blocking=True)
	
	if rank == 0:
		print('Model Summary: ')
		summary(model,input_size=(1,256,256))
	
	SoftIdxGPU = SoftIdx.to(device)
	SoftWeightsGPU = SoftWeights.to(device)
	ClassWeightsGPU = ClassWeightsTensor.to(device)

	if rank == 0:
		print('Creating Dataset: ')
	TrainDataset = ImageDataset(TrainDirName,True)
	ValDataset = ImageDataset(ValDirName,False)

	TrainSampler = DistributedSampler(TrainDataset,num_replicas=GPUCount,rank=rank,shuffle=True)
	ValSampler = DistributedSampler(ValDataset,num_replicas=GPUCount,rank=rank,shuffle=False)

	TrainLoader = DataLoader(TrainDataset,batch_size=BatchSize,sampler=TrainSampler,num_workers=4,pin_memory=True,prefetch_factor=2,drop_last=True,persistent_workers=True)
	ValLoader = DataLoader(ValDataset,batch_size=BatchSize,sampler=ValSampler,num_workers=4,pin_memory=True,prefetch_factor=2,drop_last=False,persistent_workers=True)

	Checkpoint = None
	if ResumeCheckpointPath is not None and ResumeCheckpointPath.exists():
		if rank == 0:
			print(f'Loaded previous W&B session from {ResumeCheckpointPath}!')
		Checkpoint = torch.load(ResumeCheckpointPath,map_location='cpu',weights_only=False)
		LoadCheckpointModelState(model, Checkpoint['ModelState'])

	DecayNames = {n for n, p in model.named_parameters() if p.ndim >= 2 and 'LearnParam' not in n and 'PseudoQuery' not in n}

	model = DDP(
		model,
		device_ids=[rank]
	)

	DecayParams = []
	NoDecayParams = []

	for name, param in model.named_parameters():
		if not param.requires_grad:
			continue
		(DecayParams if CleanName(name) in DecayNames else NoDecayParams).append(param)
	
	assert len(DecayParams) == len(DecayNames), f'weight-decay grouping failed: {len(DecayParams)} vs {len(DecayNames)}'

	if rank == 0:
		print(f'Decay Params: {len(DecayParams)} | No Decay Params: {len(NoDecayParams)}')

	OptimGroups = [
		{'params': DecayParams, 'weight_decay': 0.05},
		{'params': NoDecayParams, 'weight_decay': 0.0}
	]

	optimiser = optim.AdamW(OptimGroups, lr=TargetLR, fused=True)
	scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimiser,mode='min',factor=0.5,patience=3,threshold=1e-5)
	scaler = torch.amp.GradScaler('cuda')

	if Checkpoint is not None:	
		optimiser.load_state_dict(Checkpoint['OptimiserState'])
		scheduler.load_state_dict(Checkpoint['SchedulerState'])
		scaler.load_state_dict(Checkpoint['ScalerState'])
		ImagesSeen = Checkpoint.get('ImagesSeen', 0)
		BestLoss = Checkpoint.get('BestLoss', float('inf'))
		TrainLossHist = Checkpoint.get('TrainLossHist', [])
		ValLossHist = Checkpoint.get('ValLossHist', [])
		StepCounter = Checkpoint.get('StepCounter', 0)
		StartEpoch = Checkpoint.get('Epoch', -1) + 1
		PatienceCounter = Checkpoint.get('PatienceCounter', 0)
		del Checkpoint
	else:
		if rank == 0:
			print('Starting new session')
		ImagesSeen = 0
		BestLoss = float('inf')
		TrainLossHist = []
		ValLossHist = []
		StepCounter = 0
		StartEpoch = 0
		PatienceCounter = 0

	if rank == 0:
		print(f'Starting train session at epoch {StartEpoch+1}...')

	RunningLoss = torch.zeros(1, device=device)
	MaxPatience = 10
	MaxEpochs = 1000
	LastCompletedEpoch = StartEpoch - 1

	try:
		for epoch in range(StartEpoch, MaxEpochs):
			TrainSampler.set_epoch(epoch)
			model.train()
			AccumulationSteps = 4
			optimiser.zero_grad(set_to_none=True)

			for BatchIdx, (inputs, targets) in enumerate(TrainLoader):
				
				IsSyncStep = (BatchIdx + 1) % AccumulationSteps == 0 or (BatchIdx + 1) == len(TrainLoader)
				
				SyncContext = contextlib.nullcontext() if IsSyncStep else model.no_sync()

				GroupStart = (BatchIdx // AccumulationSteps) * AccumulationSteps
				GroupSize = min(AccumulationSteps, len(TrainLoader) - GroupStart)

				inputs = inputs.to(device, non_blocking=True)
				targets = targets.to(device, non_blocking=True)

				with SyncContext:
					with torch.amp.autocast('cuda', dtype=MPDType):
						predictions = model(inputs)
						loss = SoftCrossEntropyLoss(predictions, targets, SoftIdxGPU, SoftWeightsGPU, ClassWeightsGPU) / GroupSize
					scaler.scale(loss).backward()

				BatchLossTensor = loss.detach() * GroupSize
				RunningLoss += BatchLossTensor
				ImagesSeen += (inputs.size(0) * GPUCount)

				if IsSyncStep:
					if StepCounter < WarmupSteps:
						CurrentLR = TargetLR * ((StepCounter+1) / WarmupSteps)
						for ParamGroup in optimiser.param_groups:
							ParamGroup['lr'] = CurrentLR

					scaler.unscale_(optimiser)
					torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
					
					scaler.step(optimiser)
					scaler.update()
					optimiser.zero_grad(set_to_none=True)
					
					StepCounter += 1

					if rank == 0 and wandb_run is not None and (StepCounter % 20 == 0 or (BatchIdx + 1) == len(TrainLoader)):
						wandb_run.log({
							'train/batch_loss': BatchLossTensor.item(),
							'train/running_epoch_loss': (RunningLoss / (BatchIdx + 1)).item(),
							'train/epoch': epoch + 1,
							'train/batch': BatchIdx + 1,
							'train/epoch_progress': (BatchIdx + 1) / len(TrainLoader),
							'train/learning_rate': GetCurrentLR(optimiser),
							'train/images_seen': ImagesSeen,
							'train/step_counter': StepCounter
						}, step=StepCounter)

			model.eval()
			ValLossTotal = 0.0
			with torch.no_grad():
				for ValImgs,ValTargets in ValLoader:
					ValImgs = ValImgs.to(device,non_blocking=True)
					ValTargets = ValTargets.to(device,non_blocking=True)

					with torch.amp.autocast('cuda', dtype=MPDType):
						ValPrediction = model(ValImgs)
						ValLoss = SoftCrossEntropyLoss(ValPrediction, ValTargets, SoftIdxGPU, SoftWeightsGPU, ClassWeightsGPU)
					ValLossTotal += ValLoss.item()

					del ValPrediction, ValLoss
					
			LocalAvgLoss = ValLossTotal/len(ValLoader)
			LossTensor = torch.tensor([LocalAvgLoss],device=device)
			dist.all_reduce(LossTensor,op=dist.ReduceOp.SUM)
			GlobalAvgLoss = LossTensor.item()/GPUCount

			LocalAvgTrainLoss = (RunningLoss / len(TrainLoader)).item()
			TrainLossTensor = torch.tensor([LocalAvgTrainLoss],device=device)
			dist.all_reduce(TrainLossTensor,op=dist.ReduceOp.SUM)
			AverageTrainLoss = TrainLossTensor.item()/GPUCount

			TrainLossHist.append(AverageTrainLoss)
			ValLossHist.append(GlobalAvgLoss)

			if StepCounter > WarmupSteps:
				scheduler.step(GlobalAvgLoss)

			if rank == 0:
				print(f"Epoch: {epoch+1} | Training Loss: {AverageTrainLoss:.6f} | Validation Loss: {GlobalAvgLoss:.6f}")

			Improved = GlobalAvgLoss < BestLoss
			if Improved:
				BestLoss = GlobalAvgLoss
				PatienceCounter = 0
				if rank == 0:
					print(f"Validation Loss has decreased! Saving best model!")
			else:
				PatienceCounter += 1
				if rank == 0:
					print(f'No improvement... Patience: {PatienceCounter}/{MaxPatience}')

			LastCompletedEpoch = epoch
			Checkpoint = BuildCheckpoint(model, optimiser, scheduler, ImagesSeen, BestLoss, TrainLossHist, ValLossHist, StepCounter, LastCompletedEpoch, PatienceCounter, scaler)

			if rank == 0:
				Metadata = {
					'epoch': LastCompletedEpoch,
					'images_seen': ImagesSeen,
					'step_counter': StepCounter,
					'best_loss': BestLoss,
					'validation_loss': GlobalAvgLoss,
					'training_loss': AverageTrainLoss,
					'patience_counter': PatienceCounter
				}
				SaveCheckpoint(LATEST_CHECKPOINT_PATH, Checkpoint)

				if Improved:
					SaveCheckpoint(BEST_CHECKPOINT_PATH, Checkpoint)

				Aliases = ['latest', 'best'] if Improved else ['latest']
				UploadCheckpointArtifact(wandb_run, LATEST_CHECKPOINT_PATH, Aliases, Metadata)

				if wandb_run is not None:
					wandb_run.log({
						'epoch': epoch + 1,
						'train_loss': AverageTrainLoss,
						'val_loss': GlobalAvgLoss,
						'best_loss': BestLoss,
						'learning_rate': GetCurrentLR(optimiser),
						'images_seen': ImagesSeen,
						'step_counter': StepCounter,
						'patience_counter': PatienceCounter,
						'epoch/train_loss': AverageTrainLoss,
						'epoch/val_loss': GlobalAvgLoss,
						'epoch/best_loss': BestLoss,
						'epoch/learning_rate': GetCurrentLR(optimiser),
						'epoch/images_seen': ImagesSeen,
						'epoch/patience_counter': PatienceCounter
					}, step=StepCounter)

			if (PatienceCounter >= MaxPatience) and (StepCounter > WarmupSteps):
				if rank == 0:
					print('Patience limit reached. Stopping...')
				break
			
			RunningLoss.zero_()
			
			StopFlag = torch.zeros(1, device=device)

			if rank == 0 and (time.time() - StartTime) > MaxTime:
				StopFlag += 1

			dist.all_reduce(StopFlag, op=dist.ReduceOp.MAX)

			if StopFlag.item() > 0:
				if rank == 0:
					print('Approaching time limit!')
				raise KeyboardInterrupt

	except KeyboardInterrupt:
		if rank == 0:
			print(f"Keyboard Interrupt! Saving latest model!")
			Checkpoint = BuildCheckpoint(model, optimiser, scheduler, ImagesSeen, BestLoss, TrainLossHist, ValLossHist, StepCounter, LastCompletedEpoch, PatienceCounter, scaler)
			Metadata = {
				'epoch': LastCompletedEpoch,
				'images_seen': ImagesSeen,
				'step_counter': StepCounter,
				'best_loss': BestLoss,
				'patience_counter': PatienceCounter,
				'interrupted': True
			}
			SaveCheckpoint(LATEST_CHECKPOINT_PATH, Checkpoint)
			UploadCheckpointArtifact(wandb_run, LATEST_CHECKPOINT_PATH, ['latest'], Metadata)
	finally:
		if rank == 0 and wandb_run is not None:
			wandb_run.finish()
		dist.destroy_process_group()

if __name__ == '__main__':
	DDPTrainLoop(Weights,BatchSize,TrainDirName,ValDirName)

In [ ]:
!PYTHONUNBUFFERED=1 torchrun --nproc_per_node=2 PrismTrainKaggle.py 2>&1 | tee PrismTrainLog.txt